# 🧠 Module 23 Lab: Agent Reasoning & Planning

**Mục tiêu:** Implement và so sánh 4 reasoning strategies cho LoanBot — CoT, ReAct, ToT, và Planning.

**Không cần Claude API key:** Lab dùng mock LLM để focus vào framework và patterns.

**Sections:**
1. Chain-of-Thought Simulator
2. ReAct Loop với Tool Execution
3. Tree-of-Thought với Branch Scoring
4. Hierarchical Task Planner
5. Self-Reflection với Constitutional AI
6. Strategy Comparison & Routing
7. Practice: Complete Reasoning Pipeline

In [ ]:
import json, random, time
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Callable
from enum import Enum

random.seed(42)

# Mock LLM — simulates Claude responses without real API
class MockLLM:
    """Deterministic mock LLM for testing reasoning frameworks."""

    def complete(self, prompt: str, **kwargs) -> str:
        """Return structured response based on prompt keywords."""
        if 'borderline' in prompt.lower() or 'conflict' in prompt.lower():
            return json.dumps({'decision': 'REVIEW', 'confidence': 0.78, 'explanation': 'Borderline case với conflicting signals — cần senior review theo NĐ13/2023 Điều 15.', 'regulation_refs': ['NĐ13/2023 Điều 15']})
        score = self._extract_score(prompt)
        if score >= 700:
            return json.dumps({'decision': 'APPROVE', 'confidence': 0.94, 'explanation': f'Credit score {score} vượt ngưỡng tốt (≥700). DTI trong giới hạn. Đủ điều kiện theo TT39/2016 Điều 9.', 'regulation_refs': ['TT39/2016 Điều 9']})
        elif score < 500:
            return json.dumps({'decision': 'REJECT', 'confidence': 0.97, 'explanation': f'Credit score {score} dưới ngưỡng tối thiểu 500. Từ chối theo chính sách FinTech Corp.', 'regulation_refs': ['TT39/2016 Điều 9']})
        else:
            return json.dumps({'decision': 'REVIEW', 'confidence': 0.81, 'explanation': f'Credit score {score} ở mức borderline (500-699). Cần xem xét thêm theo NĐ13/2023.', 'regulation_refs': ['NĐ13/2023 Điều 15']})

    def _extract_score(self, text: str) -> int:
        import re
        matches = re.findall(r'(?:score|Score)[:\s]+(\d{3})', text)
        return int(matches[0]) if matches else 600

    def score_branch(self, branch_text: str) -> float:
        """Score a ToT branch 0-10."""
        if 'REVIEW' in branch_text: return round(random.uniform(7.5, 9.0), 1)
        if 'APPROVE' in branch_text: return round(random.uniform(5.0, 7.5), 1)
        if 'REJECT' in branch_text: return round(random.uniform(3.0, 6.5), 1)
        return 5.0

llm = MockLLM()
print('✅ MockLLM ready')

## Section 1: Chain-of-Thought Simulator

Compare accuracy: Direct Decision vs Zero-shot CoT vs Few-shot CoT trên borderline cases.

In [ ]:
@dataclass
class LoanCase:
    customer_id: str
    credit_score: int
    dti: float
    employment_months: int
    loan_vnd: int
    expert_decision: str   # ground truth


def direct_decision(case: LoanCase) -> str:
    """Simple threshold-based decision — no reasoning."""
    if case.credit_score >= 650 and case.dti <= 0.42:
        return 'APPROVE'
    elif case.credit_score < 500:
        return 'REJECT'
    else:
        return 'REVIEW'


def cot_decision(case: LoanCase, style: str = 'few_shot') -> Tuple[str, str]:
    """CoT decision with reasoning chain. Returns (decision, chain)."""
    # Build reasoning chain step-by-step
    chain = []

    # Step 1: Credit score
    if case.credit_score >= 700:
        cs_signal = f'Credit Score {case.credit_score}: VẪN trong ngưỡng TỐT (≥700) ✓'
        cs_weight = +2
    elif case.credit_score >= 580:
        cs_signal = f'Credit Score {case.credit_score}: BORDERLINE (580-700) ⚠️'
        cs_weight = 0
    else:
        cs_signal = f'Credit Score {case.credit_score}: DƯỚI ngưỡng (<580) ✗'
        cs_weight = -3
    chain.append(f'1. {cs_signal}')

    # Step 2: DTI
    if case.dti <= 0.35:
        dti_signal = f'DTI {case.dti}: THẤP, an toàn ✓'
        dti_weight = +1
    elif case.dti <= 0.42:
        dti_signal = f'DTI {case.dti}: CHẤP NHẬN được ⚠️'
        dti_weight = 0
    else:
        dti_signal = f'DTI {case.dti}: VƯỢT ngưỡng 0.42 ✗'
        dti_weight = -2
    chain.append(f'2. {dti_signal}')

    # Step 3: Employment
    if case.employment_months >= 24:
        emp_signal = f'Employment {case.employment_months}m: ỔN ĐỊNH ✓'
        emp_weight = +1
    elif case.employment_months >= 12:
        emp_signal = f'Employment {case.employment_months}m: CHẤP NHẬN được ⚠️'
        emp_weight = 0
    else:
        emp_signal = f'Employment {case.employment_months}m: QUÁ NGẮN ✗'
        emp_weight = -1
    chain.append(f'3. {emp_signal}')

    # Step 4: Synthesize
    total_weight = cs_weight + dti_weight + emp_weight
    chain.append(f'4. Tổng hợp: weight score = {total_weight}')

    if total_weight >= 2:
        decision = 'APPROVE'
        chain.append('5. Quyết định: APPROVE — indicators mostly positive, risk thấp')
    elif total_weight <= -2:
        decision = 'REJECT'
        chain.append('5. Quyết định: REJECT — indicators mostly negative')
    else:
        decision = 'REVIEW'
        chain.append('5. Quyết định: REVIEW — borderline, cần xem xét thêm theo NĐ13/2023')

    return decision, '\n'.join(chain)


# Test cases — borderline: harder for direct decision
TEST_CASES = [
    LoanCase('FC-001', 720, 0.28, 48, 200_000_000, 'APPROVE'),
    LoanCase('FC-002', 638, 0.41, 14, 120_000_000, 'REVIEW'),   # borderline
    LoanCase('FC-003', 450, 0.50, 6,  80_000_000,  'REJECT'),
    LoanCase('FC-004', 655, 0.43, 20, 150_000_000, 'REVIEW'),   # borderline
    LoanCase('FC-005', 610, 0.38, 36, 100_000_000, 'APPROVE'),  # borderline but good history
    LoanCase('FC-006', 590, 0.46, 8,  90_000_000,  'REJECT'),   # borderline with bad signals
    LoanCase('FC-007', 670, 0.32, 60, 300_000_000, 'APPROVE'),
    LoanCase('FC-008', 480, 0.55, 4,  50_000_000,  'REJECT'),
]

# Compare strategies
direct_correct = sum(1 for c in TEST_CASES if direct_decision(c) == c.expert_decision)
cot_correct = sum(1 for c in TEST_CASES if cot_decision(c)[0] == c.expert_decision)

print('=== Strategy Comparison ===')
print(f'Direct Decision:  {direct_correct}/{len(TEST_CASES)} = {direct_correct/len(TEST_CASES):.1%}')
print(f'CoT Few-shot:     {cot_correct}/{len(TEST_CASES)} = {cot_correct/len(TEST_CASES):.1%}')

# Show CoT trace for borderline case
borderline = TEST_CASES[1]  # FC-002
decision, chain = cot_decision(borderline)
print(f'\n=== CoT Trace for {borderline.customer_id} (Score {borderline.credit_score}) ===')
print(chain)
print(f'\nCoT Decision: {decision} | Expert: {borderline.expert_decision} | Match: {decision == borderline.expert_decision}')

## Section 2: ReAct Loop với Tool Execution

In [ ]:
@dataclass
class ReActStep:
    step_type: str   # thought | action | observation | answer
    content: str


# Mock tool registry
MOCK_CIC_DB = {
    'FC-002': {'bad_debt_count': 0, 'active_loans': 1, 'on_time_pct': 96.0, 'oldest_account_months': 36},
    'FC-004': {'bad_debt_count': 0, 'active_loans': 2, 'on_time_pct': 89.0, 'oldest_account_months': 24},
    'FC-005': {'bad_debt_count': 1, 'active_loans': 1, 'on_time_pct': 78.0, 'oldest_account_months': 48},
}
MOCK_BLACKLIST = {'FC-003', 'FC-008'}


def tool_query_cic(customer_id: str) -> dict:
    record = MOCK_CIC_DB.get(customer_id, {'bad_debt_count': 0, 'active_loans': 0, 'on_time_pct': 95.0})
    return {'status': 'ok', 'data': record}


def tool_check_blacklist(customer_id: str) -> dict:
    return {'status': 'ok', 'blacklisted': customer_id in MOCK_BLACKLIST}


TOOLS = {'query_cic': tool_query_cic, 'check_blacklist': tool_check_blacklist}


def react_loop(case: LoanCase, max_iter: int = 4) -> Tuple[dict, List[ReActStep]]:
    """Simulate ReAct loop for LoanBot."""
    trace = []
    context = {'cic': None, 'blacklisted': None}

    for iteration in range(max_iter):
        # THOUGHT: assess current information
        thought = _generate_thought(case, context, iteration)
        trace.append(ReActStep('thought', thought))

        # Decide next action
        action = _decide_action(case, context)

        if action is None:  # enough info — generate final answer
            answer = _generate_answer(case, context)
            trace.append(ReActStep('answer', json.dumps(answer)))
            return answer, trace

        # ACTION
        tool_name, tool_args = action
        trace.append(ReActStep('action', f'{tool_name}({json.dumps(tool_args)})'))

        # OBSERVATION
        result = TOOLS[tool_name](**tool_args)
        trace.append(ReActStep('observation', json.dumps(result)))

        # Update context
        if tool_name == 'query_cic': context['cic'] = result['data']
        if tool_name == 'check_blacklist': context['blacklisted'] = result['blacklisted']

    # max_iter reached without answer
    answer = {'decision': 'REVIEW', 'confidence': 0.5, 'explanation': 'Max iterations reached — human review required', 'regulation_refs': []}
    trace.append(ReActStep('answer', json.dumps(answer)))
    return answer, trace


def _generate_thought(case, ctx, iteration):
    if iteration == 0:
        return f'Score {case.credit_score}, DTI {case.dti}, Employment {case.employment_months}m. Cần kiểm tra blacklist và CIC trước khi quyết định.'
    parts = []
    if ctx['blacklisted'] is not None:
        parts.append(f'Blacklist: {"YES - MUST REJECT" if ctx["blacklisted"] else "clear"}.')
    if ctx['cic'] is not None:
        cic = ctx['cic']
        parts.append(f'CIC: {cic["bad_debt_count"]} nợ xấu, {cic["on_time_pct"]}% on-time.')
    if ctx['blacklisted'] and ctx['cic']:
        parts.append('Đã có đủ thông tin để quyết định.')
    return ' '.join(parts) if parts else 'Đang thu thập thêm thông tin...'


def _decide_action(case, ctx):
    if ctx['blacklisted'] is None: return ('check_blacklist', {'customer_id': case.customer_id})
    if ctx['blacklisted']: return None  # blacklisted — enough to reject
    if ctx['cic'] is None: return ('query_cic', {'customer_id': case.customer_id})
    return None  # have all info


def _generate_answer(case, ctx):
    if ctx.get('blacklisted'):
        return {'decision': 'REJECT', 'confidence': 0.99, 'explanation': 'Khách hàng trong blacklist FinTech Corp.', 'regulation_refs': ['FinTech Corp Policy']}

    cic = ctx.get('cic', {})
    cot_decision_result, chain = cot_decision(case)

    # Adjust based on CIC
    bad_debt = cic.get('bad_debt_count', 0)
    on_time = cic.get('on_time_pct', 95.0)

    if bad_debt > 0 and on_time < 85:
        decision = 'REJECT'
        reason = f'CIC: {bad_debt} nợ xấu, chỉ {on_time}% on-time — không đủ điều kiện.'
    elif cot_decision_result == 'REVIEW' and bad_debt == 0 and on_time >= 95:
        decision = 'APPROVE'  # CIC good compensates borderline
        reason = f'Mặc dù borderline trên một số tiêu chí, CIC xuất sắc ({on_time}% on-time, 0 nợ xấu) → APPROVE.'
    else:
        decision = cot_decision_result
        reason = f'CoT reasoning: {chain.split(chr(10))[-1]}. CIC: {on_time}% on-time.'

    conf = 0.95 if bad_debt == 0 else 0.85
    return {'decision': decision, 'confidence': conf, 'explanation': reason, 'regulation_refs': ['TT39/2016 Điều 9']}


# Run ReAct on borderline cases
print('=== ReAct Loop Results ===')
for case in TEST_CASES[:5]:
    answer, trace = react_loop(case)
    n_steps = len(trace)
    match = '✅' if answer['decision'] == case.expert_decision else '❌'
    print(f'{case.customer_id} (score={case.credit_score}): {answer["decision"]} conf={answer["confidence"]} steps={n_steps} expert={case.expert_decision} {match}')

# Detailed trace for FC-002
print('\n=== ReAct Trace: FC-002 ===')
_, trace = react_loop(TEST_CASES[1])
for step in trace:
    prefix = {'thought': '💭', 'action': '⚡', 'observation': '👁', 'answer': '✅'}[step.step_type]
    print(f'{prefix} [{step.step_type.upper()}]: {step.content[:100]}')

## Section 3: Tree-of-Thought với Branch Scoring

In [ ]:
@dataclass
class ToTBranch:
    name: str
    perspective: str
    reasoning: str
    decision: str
    score: float = 0.0


def generate_tot_branches(case: LoanCase) -> List[ToTBranch]:
    """Generate 3 reasoning branches for a conflicted case."""
    branches = [
        ToTBranch(
            name='Conservative',
            perspective='Rủi ro-ưu tiên: bất kỳ weakness nào → REJECT/REVIEW',
            reasoning=(
                f'Credit Score {case.credit_score}: {"OK" if case.credit_score >= 650 else "WEAK"}.\n'
                f'DTI {case.dti}: {"EXCEEDS" if case.dti > 0.40 else "OK"} threshold.\n'
                f'Employment {case.employment_months}m: {"SHORT" if case.employment_months < 24 else "STABLE"}.\n'
                'Conservative view: ANY weakness → không approve.'
            ),
            decision='REJECT' if sum([case.credit_score < 650, case.dti > 0.40, case.employment_months < 12]) >= 2 else 'REVIEW'
        ),
        ToTBranch(
            name='Moderate',
            perspective='Balanced: weigh tất cả factors, compensating strengths',
            reasoning=(
                f'Credit Score {case.credit_score}: {"positive" if case.credit_score >= 620 else "concern"}.\n'
                f'DTI {case.dti}: {"manageable" if case.dti <= 0.45 else "high"}.\n'
                f'Employment {case.employment_months}m: {"strong" if case.employment_months >= 12 else "weak"}.\n'
                'Balanced: no single dealbreaker, manage risk with conditions.'
            ),
            decision='APPROVE' if case.credit_score >= 640 and case.dti <= 0.43 and case.employment_months >= 12 else 'REVIEW'
        ),
        ToTBranch(
            name='Progressive',
            perspective='Growth-ưu tiên: borderline OK nếu overall profile positive',
            reasoning=(
                f'Total profile: score {case.credit_score}, DTI {case.dti}, emp {case.employment_months}m.\n'
                'Progressive view: if no hard violations, approve to build customer relationship.'
            ),
            decision='APPROVE' if case.credit_score >= 580 and case.dti <= 0.50 else 'REVIEW'
        ),
    ]
    return branches


def score_branches(branches: List[ToTBranch], case: LoanCase) -> List[ToTBranch]:
    """Score branches based on policy adherence and evidence quality."""
    for branch in branches:
        score = 5.0
        # Reward citing specific numbers
        if str(case.credit_score) in branch.reasoning: score += 1.0
        if str(case.dti) in branch.reasoning: score += 0.5
        # Reward REVIEW for genuinely borderline (580-680)
        if 580 <= case.credit_score <= 680 and branch.decision == 'REVIEW': score += 2.0
        # Penalize REJECT without extreme bad signals
        if branch.decision == 'REJECT' and case.credit_score >= 600: score -= 1.5
        # Penalize APPROVE if DTI exceeds threshold
        if branch.decision == 'APPROVE' and case.dti > 0.42: score -= 1.0
        branch.score = round(min(10.0, max(0.0, score)), 1)
    return sorted(branches, key=lambda b: b.score, reverse=True)


def tot_decision(case: LoanCase, beam_k: int = 1) -> Tuple[str, List[ToTBranch]]:
    """Tree-of-Thought: generate branches, score, select top-k, return best."""
    branches = generate_tot_branches(case)
    scored = score_branches(branches, case)
    # Beam: keep top-k (for k=1: just best)
    selected = scored[:beam_k]
    return selected[0].decision, scored


# Test on conflicted cases
conflicted_cases = [
    LoanCase('CONFLICT-1', 655, 0.43, 20, 150_000_000, 'REVIEW'),  # good score, high DTI
    LoanCase('CONFLICT-2', 638, 0.35, 14, 120_000_000, 'REVIEW'),  # borderline score, OK DTI
    LoanCase('CONFLICT-3', 680, 0.46, 36, 200_000_000, 'REVIEW'),  # good emp, DTI exceed
]

print('=== Tree-of-Thought Results ===')
for case in conflicted_cases:
    decision, branches = tot_decision(case)
    print(f'\n{case.customer_id} (score={case.credit_score}, dti={case.dti}, emp={case.employment_months}m):')
    for b in branches:
        marker = '← SELECTED' if b == branches[0] else ''
        print(f'  [{b.score:4.1f}] {b.name:15s}: {b.decision} {marker}')
    print(f'  ToT Decision: {decision} | Expert: {case.expert_decision} | Match: {"✅" if decision == case.expert_decision else "❌"}')

## Section 4: Hierarchical Task Planner

In [ ]:
class TaskStatus(Enum):
    PENDING = 'pending'
    COMPLETED = 'completed'
    FAILED = 'failed'
    SKIPPED = 'skipped'

@dataclass
class PlanStep:
    step_id: str
    name: str
    tool: str
    params: dict
    required: bool = True
    depends_on: List[str] = field(default_factory=list)
    failure_strategy: str = 'abort'  # abort | skip | escalate
    status: TaskStatus = TaskStatus.PENDING
    result: Optional[dict] = None


# Tool implementations
def mock_executor(tool: str, params: dict) -> dict:
    """Simulate tool execution with occasional failures."""
    time.sleep(0.01)  # simulate latency
    if tool == 'check_blacklist':
        bl = params.get('customer_id') in MOCK_BLACKLIST
        if bl: return {'status': 'ok', 'blacklisted': True}  # abort if blacklisted
        return {'status': 'ok', 'blacklisted': False}
    elif tool == 'query_cic':
        return {'status': 'ok', 'data': MOCK_CIC_DB.get(params.get('customer_id'), {'bad_debt_count': 0, 'on_time_pct': 95.0})}
    elif tool == 'verify_employment':
        # Simulate 20% timeout failure rate
        if random.random() < 0.2: return {'status': 'error', 'error': 'employer_db_timeout'}
        return {'status': 'ok', 'verified': True, 'months': params.get('months', 12)}
    elif tool == 'verify_income':
        return {'status': 'ok', 'monthly_income': 25_000_000}
    elif tool == 'appraise_collateral':
        return {'status': 'ok', 'appraised_value': 1_200_000_000}
    elif tool == 'make_decision':
        return {'status': 'ok', 'decision': 'APPROVE', 'confidence': 0.90}
    return {'status': 'error', 'error': 'unknown tool'}


class LoanApprovalPlanner:
    def create_plan(self, case: LoanCase, has_collateral: bool = False) -> List[PlanStep]:
        plan = [
            PlanStep('S1', 'Check Blacklist', 'check_blacklist', {'customer_id': case.customer_id}, required=True, failure_strategy='abort'),
            PlanStep('S2', 'Query CIC', 'query_cic', {'customer_id': case.customer_id}, depends_on=['S1'], failure_strategy='escalate'),
            PlanStep('S3', 'Verify Employment', 'verify_employment', {'customer_id': case.customer_id, 'months': case.employment_months}, depends_on=['S1'], failure_strategy='skip'),
        ]
        if case.loan_vnd > 500_000_000:
            plan.append(PlanStep('S4', 'Income Verification', 'verify_income', {'customer_id': case.customer_id}, depends_on=['S2', 'S3'], required=True, failure_strategy='abort'))
        if has_collateral:
            plan.append(PlanStep('S5', 'Collateral Appraisal', 'appraise_collateral', {'customer_id': case.customer_id}, depends_on=['S1'], failure_strategy='escalate'))
        plan.append(PlanStep('S_final', 'Generate Decision', 'make_decision', {}, depends_on=[s.step_id for s in plan], failure_strategy='abort'))
        return plan

    def execute(self, plan: List[PlanStep]) -> dict:
        results = {}
        for step in plan:
            deps_ok = all(results.get(d, {}).get('status') == 'ok' for d in step.depends_on)
            if not deps_ok:
                step.status = TaskStatus.SKIPPED
                continue
            result = mock_executor(step.tool, step.params)
            if result.get('status') == 'ok':
                step.status = TaskStatus.COMPLETED
                results[step.step_id] = result
                # Hard abort: blacklisted
                if step.tool == 'check_blacklist' and result.get('blacklisted'):
                    return {'decision': 'REJECT', 'confidence': 0.99, 'reason': 'Customer blacklisted'}
            else:
                step.status = TaskStatus.FAILED
                if step.failure_strategy == 'abort' and step.required:
                    return {'decision': 'REJECT', 'confidence': 0.95, 'reason': f'Critical step failed: {step.name}'}
                elif step.failure_strategy == 'escalate':
                    return {'decision': 'REVIEW', 'confidence': 0.6, 'reason': f'Escalate: {step.name} needs manual verification'}
        return results.get('S_final', {'decision': 'REVIEW', 'confidence': 0.5, 'reason': 'Plan incomplete'})

    def print_plan(self, plan: List[PlanStep], result: dict):
        print(f'Plan execution result: {result.get("decision", result)}')
        for s in plan:
            icon = {'completed': '✅', 'failed': '❌', 'skipped': '⏭', 'pending': '⏳'}[s.status.value]
            print(f'  {icon} {s.step_id}: {s.name} [{s.status.value}]')


planner = LoanApprovalPlanner()

# Test 1: Normal case
print('=== Case 1: Standard loan (200M) ===')
case1 = LoanCase('FC-002', 638, 0.41, 14, 200_000_000, 'REVIEW')
plan1 = planner.create_plan(case1)
result1 = planner.execute(plan1)
planner.print_plan(plan1, result1)

print('\n=== Case 2: Large loan with collateral (800M) ===')
case2 = LoanCase('FC-007', 670, 0.32, 60, 800_000_000, 'APPROVE')
plan2 = planner.create_plan(case2, has_collateral=True)
result2 = planner.execute(plan2)
planner.print_plan(plan2, result2)

print('\n=== Case 3: Blacklisted customer ===')
case3 = LoanCase('FC-003', 450, 0.50, 6, 80_000_000, 'REJECT')
plan3 = planner.create_plan(case3)
result3 = planner.execute(plan3)
planner.print_plan(plan3, result3)

## Section 5: Self-Reflection với Constitutional AI

In [ ]:
PRINCIPLES = [
    ('P1', 'Quyết định không được dựa trên province/vùng miền (geographic discrimination)'),
    ('P2', 'Phải cite ít nhất 1 regulation reference (TT39/2016 hoặc NĐ13/2023)'),
    ('P3', 'Confidence phải phù hợp với evidence — không >0.95 trên borderline case'),
    ('P4', 'Explanation phải mention tất cả negative factors, không chỉ positive'),
    ('P5', 'REVIEW decision phải specify tài liệu cần bổ sung'),
]


@dataclass
class Critique:
    principle_id: str
    status: str   # PASS | FAIL
    reason: str


def critique_decision(decision: dict, case: LoanCase) -> List[Critique]:
    """Apply Constitutional AI principles to audit a loan decision."""
    critiques = []
    explanation = decision.get('explanation', '')
    confidence = decision.get('confidence', 0.0)
    refs = decision.get('regulation_refs', [])
    dec = decision.get('decision', '')

    # P1: Geographic discrimination check
    geo_terms = ['province', 'tỉnh', 'miền', 'region', 'nông thôn', 'thành thị']
    geo_in_explanation = any(t in explanation.lower() for t in geo_terms)
    critiques.append(Critique('P1', 'FAIL' if geo_in_explanation else 'PASS',
                              'Geographic term found in explanation — potential discrimination' if geo_in_explanation else 'No geographic discrimination detected'))

    # P2: Regulation reference
    has_ref = len(refs) > 0 and any(r.startswith('TT') or r.startswith('NĐ') for r in refs)
    critiques.append(Critique('P2', 'PASS' if has_ref else 'FAIL',
                              'Regulation reference present' if has_ref else 'Missing regulation reference — required by policy'))

    # P3: Confidence calibration
    is_borderline = 580 <= case.credit_score <= 680
    overconfident = is_borderline and confidence > 0.95
    critiques.append(Critique('P3', 'FAIL' if overconfident else 'PASS',
                              f'Overconfident ({confidence}) on borderline case (score={case.credit_score})' if overconfident else 'Confidence appropriate'))

    # P4: Negative factors mentioned
    has_negatives = any(w in explanation.lower() for w in ['⚠️', 'borderline', 'cần xem xét', 'tuy nhiên', 'nhưng', 'dù'])
    is_approve = dec == 'APPROVE'
    missing_negatives = is_approve and is_borderline and not has_negatives
    critiques.append(Critique('P4', 'FAIL' if missing_negatives else 'PASS',
                              'APPROVE on borderline case without mentioning any negative signals' if missing_negatives else 'Explanation balanced'))

    # P5: REVIEW must specify documents
    if dec == 'REVIEW':
        doc_terms = ['tài liệu', 'chứng minh', 'bổ sung', 'xác nhận', 'giấy tờ', 'hợp đồng']
        has_docs = any(t in explanation.lower() for t in doc_terms)
        critiques.append(Critique('P5', 'PASS' if has_docs else 'FAIL',
                                  'Documents specified' if has_docs else 'REVIEW decision missing required documents specification'))
    else:
        critiques.append(Critique('P5', 'PASS', 'N/A — not a REVIEW decision'))

    return critiques


def self_reflect(decision: dict, case: LoanCase) -> dict:
    """Apply critique and revise decision if issues found."""
    critiques = critique_decision(decision, case)
    failures = [c for c in critiques if c.status == 'FAIL']

    if not failures:
        decision['_critique'] = 'PASS — no issues'
        return decision

    # Revise based on failures
    revised = dict(decision)
    revised['_critique'] = f'FAIL on {[f.principle_id for f in failures]}'

    if any(f.principle_id == 'P2' for f in failures):
        revised['regulation_refs'] = ['TT39/2016 Điều 9']  # add missing ref
    if any(f.principle_id == 'P3' for f in failures):
        revised['confidence'] = min(0.88, decision['confidence'])  # recalibrate
    if any(f.principle_id == 'P4' for f in failures):
        revised['explanation'] += f' ⚠️ Lưu ý: credit score {case.credit_score} ở mức borderline, DTI {case.dti} cần theo dõi.'
    if any(f.principle_id == 'P5' for f in failures):
        revised['explanation'] += ' Yêu cầu: (1) Bảng lương 3 tháng gần nhất; (2) Xác nhận việc làm từ employer; (3) Sao kê tài khoản 6 tháng.'

    return revised


# Test self-reflection
print('=== Self-Reflection Test ===')
borderline_case = TEST_CASES[1]  # FC-002, score 638

# Deliberately flawed decision (missing ref, overconfident, no negatives)
flawed = {
    'decision': 'APPROVE',
    'confidence': 0.96,
    'explanation': 'Khách hàng có profile tốt, employment ổn định.',
    'regulation_refs': []
}

critiques = critique_decision(flawed, borderline_case)
print('\nInitial decision critiques:')
for c in critiques:
    icon = '✅' if c.status == 'PASS' else '❌'
    print(f'  {icon} {c.principle_id}: {c.reason}')

revised = self_reflect(flawed, borderline_case)
print(f'\nRevised decision:')
print(f'  Decision: {revised["decision"]} | Confidence: {revised["confidence"]}')
print(f'  Refs: {revised["regulation_refs"]}')
print(f'  Explanation: {revised["explanation"][:150]}...')
print(f'  Critique: {revised["_critique"]}')

## Section 6: Strategy Comparison & Routing

In [ ]:
def classify_case(case: LoanCase) -> str:
    """Route case to appropriate reasoning strategy."""
    # Clear REJECT: any blacklist signal or very low score
    if case.credit_score < 500 or case.dti > 0.55:
        return 'direct'
    # Clear APPROVE: all signals positive
    if case.credit_score >= 700 and case.dti <= 0.35 and case.employment_months >= 24:
        return 'direct'
    # Conflicting signals: use ToT
    positive = sum([case.credit_score >= 650, case.dti <= 0.40, case.employment_months >= 24])
    negative = sum([case.credit_score < 620, case.dti > 0.42, case.employment_months < 12])
    if positive >= 1 and negative >= 1:
        return 'tot'
    # Borderline but no conflict: use ReAct (need external info)
    if 580 <= case.credit_score <= 680:
        return 'react'
    # Default: CoT
    return 'cot'


def route_and_decide(case: LoanCase) -> Tuple[str, str, str]:
    """Route to strategy, execute, return (decision, strategy, expert)."""
    strategy = classify_case(case)

    if strategy == 'direct':
        decision = direct_decision(case)
    elif strategy == 'cot':
        decision, _ = cot_decision(case)
    elif strategy == 'react':
        answer, _ = react_loop(case)
        decision = answer['decision']
    elif strategy == 'tot':
        decision, _ = tot_decision(case)
    else:
        decision = 'REVIEW'

    return decision, strategy, case.expert_decision


print('=== Smart Routing Strategy Comparison ===')
print(f'{"Case":<12}{"Score":<8}{"DTI":<7}{"Emp":<8}{"Strategy":<10}{"Decision":<10}{"Expert":<10}{"Match"}')
print('-' * 75)

correct = 0
strategy_counts = {}
for case in TEST_CASES:
    decision, strategy, expert = route_and_decide(case)
    match = '✅' if decision == expert else '❌'
    if decision == expert: correct += 1
    strategy_counts[strategy] = strategy_counts.get(strategy, 0) + 1
    print(f'{case.customer_id:<12}{case.credit_score:<8}{case.dti:<7}{case.employment_months:<8}{strategy:<10}{decision:<10}{expert:<10}{match}')

print(f'\nOverall accuracy: {correct}/{len(TEST_CASES)} = {correct/len(TEST_CASES):.1%}')
print(f'Strategy distribution: {strategy_counts}')

## Section 7: Practice — Complete Reasoning Pipeline

**Bài tập:** Implement `ReasoningOrchestrator` kết hợp tất cả strategies với self-reflection.

In [ ]:
class ReasoningOrchestrator:
    """Complete reasoning pipeline with routing + strategy + self-reflection."""

    def process(self, case: LoanCase) -> dict:
        """TODO: Implement full pipeline:
        1. Classify case → choose strategy
        2. Execute chosen strategy
        3. Apply self-reflection critique
        4. Return final decision with metadata
        """
        # YOUR CODE HERE
        pass


# --- SOLUTION ---
class ReasoningOrchestratorSolution:

    def process(self, case: LoanCase) -> dict:
        # Step 1: Route
        strategy = classify_case(case)

        # Step 2: Execute
        if strategy == 'direct':
            dec = direct_decision(case)
            conf = 0.97 if dec in ('APPROVE', 'REJECT') else 0.80
            initial = {'decision': dec, 'confidence': conf, 'explanation': f'Clear case: score={case.credit_score}, dti={case.dti}', 'regulation_refs': ['TT39/2016 Điều 9']}
        elif strategy == 'cot':
            dec, chain = cot_decision(case)
            initial = {'decision': dec, 'confidence': 0.86, 'explanation': chain.split('\n')[-1], 'regulation_refs': ['TT39/2016 Điều 9']}
        elif strategy == 'react':
            initial, _ = react_loop(case)
        elif strategy == 'tot':
            dec, branches = tot_decision(case)
            initial = {'decision': dec, 'confidence': 0.82, 'explanation': f'ToT: selected {branches[0].name} branch (score {branches[0].score}/10). {branches[0].reasoning.split(chr(10))[-1]}', 'regulation_refs': ['NĐ13/2023 Điều 15']}
        else:
            initial = {'decision': 'REVIEW', 'confidence': 0.5, 'explanation': 'Unknown strategy', 'regulation_refs': []}

        # Step 3: Self-reflection
        final = self_reflect(initial, case)

        # Step 4: Add metadata
        final['_strategy'] = strategy
        final['_customer_id'] = case.customer_id
        return final


# Run complete pipeline
orchestrator = ReasoningOrchestratorSolution()

print('=== Complete Reasoning Pipeline ===')
correct = 0
for case in TEST_CASES:
    result = orchestrator.process(case)
    match = '✅' if result['decision'] == case.expert_decision else '❌'
    if result['decision'] == case.expert_decision: correct += 1
    critique_info = result.get('_critique', 'unknown')
    print(f'{case.customer_id}: {result["decision"]} (conf={result["confidence"]:.2f}) [{result["_strategy"]}] expert={case.expert_decision} {match} | critique: {critique_info}')

print(f'\nFinal accuracy: {correct}/{len(TEST_CASES)} = {correct/len(TEST_CASES):.1%}')